# Triage-IN — Phase 1 data generation (free Kaggle GPU)

| Step | Model | Where |
|---|---|---|
| 1a. Generate ~6,000 training complaints | Gemini free tier (`gemini-3.5-flash-lite`) | Google API |
| 1b. Blind second labeller; drop category disagreements, mask other disagreements | Qwen2.5-7B-Instruct | this GPU |
| 1d. Generate 350 test candidates (different model + prompts) | Qwen2.5-7B-Instruct | this GPU |

## How to run (important)
Use **Save Version → Save & Run All (Commit)**, not the interactive session.
A committed run keeps going in the background for up to 12 hours (no need to keep the tab open),
and **its output files are saved permanently** in the version's *Output* tab.

**Settings before committing:** Accelerator **GPU T4 x2**, Internet **On**,
Add-ons → Secrets → `GEMINI_API_KEY` attached to this notebook.

**Resuming:** if a run stops partway, open the old version's Output, and add it to this notebook via
*Add Input → Your Work → Notebook Output*. The first cell below finds `phase1_outputs.zip` in
`/kaggle/input` and restores it, so finished batches are skipped.

In [ ]:
import glob, os, subprocess
os.chdir("/kaggle/working")
if not os.path.isdir("triage-in"):
    subprocess.run(["git", "clone", "-q", "https://github.com/pcd24/triage-in.git"], check=True)
os.chdir("/kaggle/working/triage-in")
!git pull -q && git log --oneline -1
# Kaggle already has torch/transformers; install only what's missing.
!pip install -q -e . --no-deps && pip install -q regex google-genai==2.33.0

# Restore a previous run's outputs if one was added as input (makes every step resumable).
backups = glob.glob("/kaggle/input/**/phase1_outputs.zip", recursive=True)
if backups:
    print("Restoring", backups[0])
    !unzip -o -q {backups[0]} -d /kaggle/working/triage-in
else:
    print("No previous outputs found; starting fresh.")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
!python -c "import torch; print(torch.cuda.device_count(), 'GPU(s)')"

In [ ]:
def backup():
    # Zip everything after each step so even a partial run leaves usable, saved output.
    !cd /kaggle/working/triage-in && zip -q -r -FS /kaggle/working/phase1_outputs.zip data/raw data/interim data/manual -x 'data/*/smoke/*'
    !ls -lh /kaggle/working/phase1_outputs.zip

## 1a. Training data with Gemini (free tier, ~1 h)
Run twice: the second pass retries any batch whose reply failed to parse (cached batches are skipped).

In [ ]:
!python -m triage_in.data.generate_synthetic --yes 2>&1 | grep -v -E "httpx|AFC is enabled"
!python -m triage_in.data.generate_synthetic --yes 2>&1 | grep -v -E "httpx|AFC is enabled" | tail -3
backup()

## 1b. Independent labeller (open model on this GPU, ~1.5 h)

In [ ]:
!python -m triage_in.data.validate_labels --yes 2>&1 | grep -v -E "httpx|Downloading|Reconstructing"
!cat data/interim/validation_report.json
backup()

## 1d. Test candidates (different model + different prompts, ~15 min)

In [ ]:
!python -m triage_in.data.generate_test --yes 2>&1 | grep -v -E "httpx|Downloading|Reconstructing"
backup()

## Quick look (logged for the record)

In [ ]:
import json, random
rows = [json.loads(l) for l in open("data/interim/validated.jsonl", encoding="utf-8")]
random.seed(7)
for r in random.sample(rows, 10):
    print(f"[{r['lang_mix']} | {r['category']} / {r['sentiment']} / {r['urgency']}]  {r['text']}\n")

## Done
Download `phase1_outputs.zip` from this version's **Output** tab and send it back. Review `data/manual/test_synthetic_review.csv` (inside the zip) by hand.